In [0]:
%sql
-- =============================================================
-- Projet Retail Azure - Couche SILVER
-- Source : volume Bronze  /Volumes/retail/bronze/raw/  (fichiers déposés par ADF)
-- Cible  : tables Delta managées  retail.silver.*
-- Règles : typage, normalisation, filtres qualité,
--          dédoublonnage = dernière version par clé (ingestion_date la plus récente)
-- =============================================================

USE CATALOG retail;
USE SCHEMA silver;

-- -------------------------------------------------------------
-- 1. Clients (API JSON)
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.silver.customers
COMMENT 'Clients nettoyés - dernière version par customer_id'
AS
SELECT
  CAST(customer_id AS INT)               AS customer_id,
  INITCAP(TRIM(first_name))              AS first_name,
  INITCAP(TRIM(last_name))               AS last_name,
  LOWER(TRIM(email))                     AS email,
  TRIM(phone)                            AS phone,
  INITCAP(TRIM(city))                    AS city,
  TRY_CAST(registration_date AS DATE)    AS registration_date,
  ingestion_date,
  current_timestamp()                    AS _silver_loaded_at
FROM read_files(
  '/Volumes/retail/bronze/raw/api/customers/',
  format => 'json',
  multiLine => true
)
WHERE customer_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY ingestion_date DESC) = 1;

-- -------------------------------------------------------------
-- 2. Produits (SQL -> Parquet)
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.silver.products
COMMENT 'Produits nettoyés - dernière version par product_id'
AS
SELECT
  CAST(product_id AS INT)                AS product_id,
  TRIM(product_name)                     AS product_name,
  INITCAP(TRIM(category))                AS category,
  CAST(price AS DECIMAL(10,2))           AS price,
  ingestion_date,
  current_timestamp()                    AS _silver_loaded_at
FROM read_files('/Volumes/retail/bronze/raw/sql/products/', format => 'parquet')
WHERE product_id IS NOT NULL
  AND price >= 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY product_id ORDER BY ingestion_date DESC) = 1;

-- -------------------------------------------------------------
-- 3. Magasins (SQL -> Parquet)
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.silver.stores
COMMENT 'Magasins nettoyés - dernière version par store_id'
AS
SELECT
  CAST(store_id AS INT)                  AS store_id,
  TRIM(store_name)                       AS store_name,
  INITCAP(TRIM(location))                AS city,
  ingestion_date,
  current_timestamp()                    AS _silver_loaded_at
FROM read_files('/Volumes/retail/bronze/raw/sql/stores/', format => 'parquet')
WHERE store_id IS NOT NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY store_id ORDER BY ingestion_date DESC) = 1;

-- -------------------------------------------------------------
-- 4. Transactions (SQL -> Parquet)
-- -------------------------------------------------------------
CREATE OR REPLACE TABLE retail.silver.transactions
COMMENT 'Transactions nettoyées - dernière version par transaction_id'
AS
SELECT
  CAST(transaction_id AS INT)            AS transaction_id,
  CAST(customer_id AS INT)               AS customer_id,
  CAST(product_id AS INT)                AS product_id,
  CAST(store_id AS INT)                  AS store_id,
  CAST(quantity AS INT)                  AS quantity,
  CAST(transaction_date AS DATE)         AS transaction_date,
  ingestion_date,
  current_timestamp()                    AS _silver_loaded_at
FROM read_files('/Volumes/retail/bronze/raw/sql/transactions/', format => 'parquet')
WHERE transaction_id IS NOT NULL
  AND quantity > 0
QUALIFY ROW_NUMBER() OVER (PARTITION BY transaction_id ORDER BY ingestion_date DESC) = 1;

-- -------------------------------------------------------------
-- 5. Contrôles qualité
--    Attendu : customers 50 | products 10 | stores 5 | transactions 30
--    orphan_* doivent être à 0
-- -------------------------------------------------------------
SELECT
  (SELECT COUNT(*) FROM retail.silver.customers)    AS customers,
  (SELECT COUNT(*) FROM retail.silver.products)     AS products,
  (SELECT COUNT(*) FROM retail.silver.stores)       AS stores,
  (SELECT COUNT(*) FROM retail.silver.transactions) AS transactions,
  (SELECT COUNT(*) FROM retail.silver.transactions t
     LEFT ANTI JOIN retail.silver.customers c ON t.customer_id = c.customer_id) AS orphan_customers,
  (SELECT COUNT(*) FROM retail.silver.transactions t
     LEFT ANTI JOIN retail.silver.products p ON t.product_id = p.product_id)    AS orphan_products,
  (SELECT COUNT(*) FROM retail.silver.transactions t
     LEFT ANTI JOIN retail.silver.stores s ON t.store_id = s.store_id)          AS orphan_stores;